## model_based_cf_test

In [1]:
# 1. 데이터셋 다운로드
!wget -q http://files.grouplens.org/datasets/movielens/ml-100k.zip -P /content/
!unzip -qo /content/ml-100k.zip -d /content/
print('다운로드 완료')

다운로드 완료


In [2]:
# 2. 라이브러리 임포트
import math
import random
import pickle
from collections import defaultdict
from pathlib import Path
import numpy as np

In [3]:
# 3. 함수
def read_ratings(path):
    rows = []
    with open(path, "r", encoding="latin-1") as f:
        for line in f:
            user_id, item_id, rating, timestamp = line.strip().split("\t")
            rows.append((int(user_id), int(item_id), float(rating), int(timestamp)))
    return rows

def read_items(path):
    titles = {}
    genres = {}
    with open(path, "r", encoding="latin-1") as f:
        for line in f:
            parts = line.rstrip("\n").split("|")
            iid = int(parts[0])
            titles[iid] = parts[1]
            genre_vec = np.array([int(x) for x in parts[5:24]], dtype=np.float32)
            norm = np.linalg.norm(genre_vec)
            genres[iid] = genre_vec / norm if norm > 0 else genre_vec
    return titles, genres

In [4]:
# 4. MFRecommender 클래스
class MFRecommender:
    def __init__(self, n_factors=20, lr=0.005, reg=0.02, n_epochs=30,
                 min_candidate_popularity=1, seed=42):
        self.n_factors = n_factors
        self.lr = lr
        self.reg = reg
        self.n_epochs = n_epochs
        self.min_candidate_popularity = min_candidate_popularity
        self.seed = seed

    def fit(self, train_rows):
        self.users = sorted({u for u, _, _, _ in train_rows})
        self.items = sorted({i for _, i, _, _ in train_rows})
        self.user_to_idx = {u: idx for idx, u in enumerate(self.users)}
        self.item_to_idx = {i: idx for idx, i in enumerate(self.items)}
        n_users, n_items = len(self.users), len(self.items)

        self.user_rated_items = defaultdict(list)
        self.item_popularity  = defaultdict(int)
        rating_sum = 0.0
        for user_id, item_id, rating, _ in train_rows:
            self.user_rated_items[user_id].append((item_id, rating))
            self.item_popularity[item_id] += 1
            rating_sum += rating
        self.global_mean = rating_sum / len(train_rows) if train_rows else 3.0

        rng = np.random.default_rng(self.seed)
        self.U   = rng.normal(0, 0.01, (n_users, self.n_factors)).astype(np.float32)
        self.V   = rng.normal(0, 0.01, (n_items, self.n_factors)).astype(np.float32)
        self.b_u = np.zeros(n_users, dtype=np.float32)
        self.b_i = np.zeros(n_items, dtype=np.float32)

        train_list = [(self.user_to_idx[u], self.item_to_idx[i], r) for u, i, r, _ in train_rows]
        rng_shuffle = random.Random(self.seed)
        for epoch in range(self.n_epochs):
            rng_shuffle.shuffle(train_list)
            sq_err = 0.0
            for uidx, iidx, rating in train_list:
                pred = (self.global_mean + self.b_u[uidx] + self.b_i[iidx]
                        + float(self.U[uidx] @ self.V[iidx]))
                err    = rating - pred
                sq_err += err * err
                v_old = self.V[iidx].copy()
                self.V[iidx]   += self.lr * (err * self.U[uidx] - self.reg * self.V[iidx])
                self.U[uidx]   += self.lr * (err * v_old        - self.reg * self.U[uidx])
                self.b_u[uidx] += self.lr * (err - self.reg * self.b_u[uidx])
                self.b_i[iidx] += self.lr * (err - self.reg * self.b_i[iidx])
            if (epoch + 1) % 5 == 0:
                print(f"  epoch {epoch+1:>2}/{self.n_epochs}  train RMSE={math.sqrt(sq_err/len(train_list)):.4f}")
        return self

    def predict(self, user_id, item_id):
        has_u = user_id in self.user_to_idx
        has_i = item_id in self.item_to_idx
        if not has_u and not has_i: return self._clip(self.global_mean)
        if not has_u: return self._clip(self.global_mean + self.b_i[self.item_to_idx[item_id]])
        if not has_i: return self._clip(self.global_mean + self.b_u[self.user_to_idx[user_id]])
        uidx, iidx = self.user_to_idx[user_id], self.item_to_idx[item_id]
        return self._clip(self.global_mean + self.b_u[uidx] + self.b_i[iidx]
                          + float(self.U[uidx] @ self.V[iidx]))

    def recommend(self, user_id, top_n=10):
        if user_id not in self.user_to_idx:
            return self.popular_items(top_n)
        seen = {iid for iid, _ in self.user_rated_items[user_id]}
        candidates = [iid for iid in self.items
                      if iid not in seen
                      and self.item_popularity.get(iid, 0) >= self.min_candidate_popularity]
        scored = [(iid, self.predict(user_id, iid)) for iid in candidates]
        scored.sort(key=lambda x: (x[1], self.item_popularity.get(x[0], 0)), reverse=True)
        return scored[:top_n]

    def popular_items(self, top_n=10):
        sorted_items = sorted(self.item_popularity.keys(),
            key=lambda iid: (self.item_popularity[iid], self.item_mean(iid)), reverse=True)
        return [(iid, self.item_mean(iid)) for iid in sorted_items[:top_n]]

    def item_mean(self, item_id):
        if item_id not in self.item_to_idx: return self.global_mean
        return self._clip(self.global_mean + self.b_i[self.item_to_idx[item_id]])

    @staticmethod
    def _clip(value): return min(5.0, max(1.0, float(value)))

In [5]:
# 5. 평가 함수
def rating_metrics(model, rows):
    sq_errors, abs_errors = [], []
    for user_id, item_id, rating, _ in rows:
        pred = model.predict(user_id, item_id)
        e = rating - pred
        sq_errors.append(e * e)
        abs_errors.append(abs(e))
    return {
        "rmse": math.sqrt(sum(sq_errors) / len(sq_errors)),
        "mae":  sum(abs_errors) / len(abs_errors),
    }

def topn_metrics(model, eval_rows, genres, top_n=10, relevant_threshold=4.0):
    eval_users = sorted({u for u, _, _, _ in eval_rows if u in model.user_to_idx})
    relevant_by_user = defaultdict(set)
    for user_id, item_id, rating, _ in eval_rows:
        if rating >= relevant_threshold:
            relevant_by_user[user_id].add(item_id)
    all_recommended  = set()
    novelty_scores, diversity_scores = [], []
    precisions, recalls, f1_scores = [], [], []
    total_interactions = sum(model.item_popularity.values())
    for user_id in eval_users:
        recs      = model.recommend(user_id, top_n=top_n)
        rec_items = [iid for iid, _ in recs]
        all_recommended.update(rec_items)
        for iid in rec_items:
            pop = max(model.item_popularity.get(iid, 1), 1)
            novelty_scores.append(-math.log2(pop / total_interactions))
        if len(rec_items) >= 2:
            vecs = [genres[iid] for iid in rec_items if iid in genres]
            if len(vecs) >= 2:
                sim_sum, pair_count = 0.0, 0
                for a in range(len(vecs)):
                    for b in range(a + 1, len(vecs)):
                        sim_sum    += float(np.dot(vecs[a], vecs[b]))
                        pair_count += 1
                avg_sim = sim_sum / pair_count if pair_count > 0 else 0.0
                diversity_scores.append(1.0 - avg_sim)
        relevant = relevant_by_user.get(user_id, set())
        if relevant:
            hits = len(set(rec_items) & relevant)
            p = hits / top_n
            r = hits / len(relevant)
            precisions.append(p)
            recalls.append(r)
            f1_scores.append((2 * p * r / (p + r)) if (p + r) > 0 else 0.0)
    return {
        "precision":       sum(precisions)      / len(precisions)      if precisions      else 0.0,
        "recall":          sum(recalls)          / len(recalls)         if recalls         else 0.0,
        "f1":              sum(f1_scores)        / len(f1_scores)       if f1_scores       else 0.0,
        "coverage":        len(all_recommended)  / len(model.items),
        "novelty":         sum(novelty_scores)   / len(novelty_scores)  if novelty_scores  else 0.0,
        "diversity":       sum(diversity_scores) / len(diversity_scores)if diversity_scores else 0.0,
        "users_evaluated": len(eval_users),
    }

def print_recommendations(model, titles, user_id, top_n=10):
    print(f"\nSample recommendations for user {user_id}")
    for rank, (item_id, score) in enumerate(model.recommend(user_id, top_n=top_n), start=1):
        title = titles.get(item_id, f"movieId={item_id}")
        print(f"{rank:2d}. {title} | predicted_rating={score:.3f}")

In [6]:
# 6. Google Drive 마운트 & model.pkl 로드
from google.colab import drive
drive.mount('/content/drive')

# train.ipynb 에서 저장한 경로와 동일하게
model_path = '/content/drive/MyDrive/mf_model/model.pkl'

with open(model_path, 'rb') as f:
    bundle = pickle.load(f)

final_model = bundle['model']
titles      = bundle['titles']
genres      = bundle['genres']
print(f"모델 로드 완료: {model_path}")

Mounted at /content/drive
모델 로드 완료: /content/drive/MyDrive/mf_model/model.pkl


In [7]:
# 7. 평가
top_n_recs     = 10
sample_user_id = 1
data_dir       = Path('/content/ml-100k')

test_rows = read_ratings(data_dir / 'ua.test')
print(f"test rows: {len(test_rows):,}")

r_m = rating_metrics(final_model, test_rows)
t_m = topn_metrics(final_model, test_rows, genres, top_n=top_n_recs)

print("\n[최종 평가] ua.test")
print("  [정확도 — 평점 예측]")
print(f"  - RMSE          : {r_m['rmse']:.4f}")
print(f"  - MAE           : {r_m['mae']:.4f}")
print("  [정확도 — 랭킹]")
print(f"  - Precision@{top_n_recs:<2} : {t_m['precision']:.4f}")
print(f"  - Recall@{top_n_recs:<2}    : {t_m['recall']:.4f}")
print(f"  - F1@{top_n_recs:<2}        : {t_m['f1']:.4f}")
print("  [Beyond-Accuracy]")
print(f"  - Coverage      : {t_m['coverage']:.4f}")
print(f"  - Novelty       : {t_m['novelty']:.4f}")
print(f"  - Diversity     : {t_m['diversity']:.4f}")
print(f"  - Users eval'd  : {t_m['users_evaluated']}")

print_recommendations(final_model, titles, sample_user_id, top_n=top_n_recs)

test rows: 9,430

[최종 평가] ua.test
  [정확도 — 평점 예측]
  - RMSE          : 0.9422
  - MAE           : 0.7438
  [정확도 — 랭킹]
  - Precision@10 : 0.0315
  - Recall@10    : 0.0541
  - F1@10        : 0.0382
  [Beyond-Accuracy]
  - Coverage      : 0.0726
  - Novelty       : 9.5344
  - Diversity     : 0.7387
  - Users eval'd  : 943

Sample recommendations for user 1
 1. Casablanca (1942) | predicted_rating=4.654
 2. Close Shave, A (1995) | predicted_rating=4.644
 3. Dr. Strangelove or: How I Learned to Stop Worrying and Love the Bomb (1963) | predicted_rating=4.528
 4. Schindler's List (1993) | predicted_rating=4.510
 5. Secrets & Lies (1996) | predicted_rating=4.505
 6. One Flew Over the Cuckoo's Nest (1975) | predicted_rating=4.493
 7. Rear Window (1954) | predicted_rating=4.476
 8. Lawrence of Arabia (1962) | predicted_rating=4.460
 9. Pather Panchali (1955) | predicted_rating=4.440
10. Third Man, The (1949) | predicted_rating=4.432
